# Modeling World Cup Scoring Average using FIFA Ranking Points & Top 5 League Roster, 2010-2022
This notebook builds on the work in the exploratory notebooks `wc_2018_exploration`, `wc_2022_exploration`, and `wc_2022_player_level` to fully explore how well FIFA rank at the beginning of the tournament (a proxy for the nation's performance in past international matches) and number of rostered players from the Top 5 Leagues of European Club Football (a proxy for player strength) can predict World Cup Scoring Average, defined as the team's total goals scored - total goals conceded, divided by games played, across the tournament. A few notes:
- Initial exploration suggested that using FIFA Ranking Points, rather than the ordinal rank, may give better results, so that is the approach that will be pursued here.
- Much of the code here is copied, with small adaptations, from the exploratory notebooks referenced above; the code is more thoroughly documented with longer markdown explanations there.


In [1]:
# Import statements:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

## Setup: Creating the Y Matrix
The below code is adapted from the `wc_2022_exploration` notebook. It first creates a 2D array, in which each row represents the performance of a particular nation during a particular World Cup for the four tournaments we are studying. Each row contains a team name, total number of World Cup matches played by that team that year, the team's total goals scored and total goals conceded across that tournament, and their scoring average for that tournament, which is derived from the prior three values.

This array is grouped chronologically by tournament year, such that rows 0-31 correspond to 2010 results, 32-63 are 2014, 64-95 are 2018, and 96-127 are 2022. Because this data is being aggregated from a variety of sources, the ordering of team names within a particular year's data isn't consistent across all four sets of results. Therefore, the precise arrangement of teams in each of the four segments of the data will need to be noted and retained for later use to ensure that the X Matrix values are properly ordered. Finally, from this 2D array of detailed results we derive a 1D vector of the 128 scoring average values, which will be used as the outcome variable for later modeling efforts.

For the 2022 and 2018 results, we're working with `.csv` files sourced from Kaggle, as specified in our `citations.txt`; for the 2014 and 2010 results, we've transcribed the necessary data from <https://fbref.com>.


### Tournament results arrays

First, we create the arrays with the relevant data from each of the 2010, 2014, 2018, and 2022 World Cups. Each row of each array contains the following data: team name, matches played, total goals scored, total goals conceded, and scoring average, where scoring average is defined as total goals scored minus total goals conceded, divided by matches played.

In [ ]:
# Each row of each team_results array contains the following data:
# [team_name, matches_played, goals_scored, goals_conceded, scoring_average]

# The following data is transcribed from fbref.com's 2010 World Cup data
# Note the circumflex in Côte d'Ivoire! It's needed to match the style in the FIFA rankings csv.
team_list_2010 = ['Algeria', 'Argentina', 'Australia', 'Brazil', 'Cameroon', 'Chile', "Côte d'Ivoire",
                    'Denmark', 'England', 'France', 'Germany', 'Ghana', 'Greece', 'Honduras', 'Italy',
                    'Japan', 'Korea DPR', 'Korea Republic', 'Mexico', 'Netherlands', 'New Zealand',
                    'Nigeria', 'Paraguay', 'Portugal', 'Serbia', 'Slovakia', 'Slovenia', 'South Africa',
                    'Spain', 'Switzerland', 'USA', 'Uruguay']

team_results_2010 = [
    ['Algeria', 3, 0, 2, (2-0)/3],
    ['Argentina', 5, 9, 6, (9-6)/5],
    ['Australia', 3, 3, 6, (3-6)/3],
    ['Brazil', 5, 9, 4, (9-5)/5],
    ['Cameroon', 3, 2, 5, (2-5)/3],
    ['Chile', 4, 3, 5, (3-5)/4],
    ["Côte d'Ivoire", 3, 4, 3, (4-3)/3],
    ['Denmark', 3, 3, 5, (3-5)/3],
    ['England', 4, 3, 5, (3-5)/4],
    ['France', 3, 1, 4, (1-4)/3],
    ['Germany', 7, 16, 5, (16-5)/7],
    ['Ghana', 5, 5, 4, (5-4)/5],
    ['Greece', 3, 2, 5, (2-5)/3],
    ['Honduras', 3, 0, 3, (0-3)/3],
    ['Italy', 3, 4, 5, (4-5)/3],
    ['Japan', 4, 4, 2, (4-2)/3],
    ['Korea DPR', 3, 1, 12, (1-12)/3],
    ['Korea Republic', 4, 6, 7, (6-7)/4],
    ['Mexico', 4, 4, 5, (4-5)/4],
    ['Netherlands', 7, 11, 6, (11-6)/7],
    ['New Zealand', 3, 2, 2, (2-2)/3],
    ['Nigeria', 3, 3, 5, (3-5)/3],
    ['Paraguay', 5, 3, 2, (3-2)/5],
    ['Portugal', 4, 7, 1, (7-1)/4],
    ['Serbia', 3, 2, 3, (2-3)/3],
    ['Slovakia', 4, 5, 7, (5-7)/4],
    ['Slovenia', 3, 3, 3, (3-3)/3],
    ['South Africa', 3, 3, 5, (3-5)/3],
    ['Spain', 7, 8, 2, (8-2)/7],
    ['Switzerland', 3, 1, 1, 0],
    ['USA', 4, 5, 5, 0],
    ['Uruguay', 7, 11, 8, (11-8)/7]
]

# The following data is transcribed from fbref.com's 2014 World Cup data.
# Note the circumflex in Côte d'Ivoire! It's needed to match the style in the FIFA rankings csv.
team_list_2014 = ['Algeria', 'Argentina', 'Australia', 'Belgium', 'Bosnia and Herzegovina',
                    'Brazil', 'Cameroon', 'Chile', 'Colombia', 'Costa Rica', "Côte d'Ivoire",
                    'Croatia', 'Ecuador', 'England', 'France', 'Germany', 'Ghana', 'Greece',
                    'Honduras', 'IR Iran', 'Italy', 'Japan', 'Korea Republic', 'Mexico',
                    'Netherlands', 'Nigeria', 'Portugal', 'Russia', 'Spain', 'Switzerland',
                    'USA', 'Uruguay']
team_results_2014 = [
    ['Algeria', 4, 7, 7, 0],
    ['Argentina', 7, 7, 4, (7-4)/7],
    ['Australia', 3, 3, 9, (3-9)/3],
    ['Belgium', 5, 6, 3, (6-3)/5],
    ['Bosnia and Herzegovina', 3, 4, 3, (4-3)/3],
    ['Brazil', 7, 11, 13, (11-13)/7],
    ['Cameroon', 3, 1, 9, (1-9)/3],
    ['Chile', 4, 6, 4, (6-4)/4],
    ['Colombia', 5, 12, 4, (12-4)/5],
    ['Costa Rica', 5, 5, 2, (5-2)/5],
    ["Côte d'Ivoire", 3, 4, 5, (4-5)/3],
    ['Croatia', 3, 5, 6, (5-6)/3],
    ['Ecuador', 3, 3, 3, (3-3)/3],
    ['England', 3, 2, 4, (2-4)/3],
    ['France', 5, 8, 3, (8-3)/5],
    ['Germany', 7, 18, 4, (18-4)/7],
    ['Ghana', 3, 4, 5, (4-5)/3],
    ['Greece', 4, 3, 5, (3-5)/4],
    ['Honduras', 3, 1, 7, (1-7)/3],
    ['IR Iran', 3, 1, 4, (1-4)/3],
    ['Italy', 3, 2, 3, (2-3)/3],
    ['Japan', 3, 2, 6, (2-6)/3],
    ['Korea Republic', 3, 3, 6, (3-6)/3],
    ['Mexico', 4, 5, 3, (5-3)/4],
    ['Netherlands', 7, 15, 4, (15-4)/7],
    ['Nigeria', 4, 3, 4, (3-4)/4],
    ['Portugal', 3, 3, 7, (3-7)/3],
    ['Russia', 3, 2, 3, (2-3)/3],
    ['Spain', 3, 4, 7, (4-7)/3],
    ['Switzerland', 4, 7, 7, 0],
    ['USA', 4, 5, 6, (5-6)/4],
    ['Uruguay', 4, 4, 6, (4-6)/4]
]

# Next we construct the 2018 array from the wc_2018 csv
wc_2018 = pd.read_csv('wc_2018.csv')
team_list_2018 = wc_2018['Team'].unique()
team_results_2018 = []
for team in team_list_2018:
    games = 0
    goals_scored = 0
    goals_allowed = 0
    for i in range(wc_2018.shape[0]):
        if wc_2018.iloc[i, 2]==team:
            games += 1
            goals_scored += wc_2018.iloc[i, 8]
            goals_allowed += wc_2018.iloc[i, 9]
    score_average = (goals_scored/games - goals_allowed/games)
    team_results_2018.append([team, games, goals_scored, goals_allowed, score_average])

# Finally we construct the 2022 array from the wc_2022 csv
wc_2022 = pd.read_csv('wc_2022.csv')
team_list_2022 = wc_2022['team1'].unique()
team_results_2022 = []
for team in team_list_2022:
    games = 0
    goals_scored = 0
    goals_allowed = 0
    for i in range(wc_2022.shape[0]):
        if wc_2022.iloc[i, 0]==team:
            games += 1
            goals_scored += wc_2022.iloc[i, 5]
            goals_allowed += wc_2022.iloc[i, 6]
        elif wc_2022.iloc[i, 1]==team:
            games += 1
            goals_scored += wc_2022.iloc[i, 6]
            goals_allowed += wc_2022.iloc[i, 5]
    score_average = (goals_scored/games - goals_allowed/games)
    team_results_2022.append([team, games, goals_scored, goals_allowed, score_average])

# Prior data exploration revealed that two of the 2022 team names need to be edited
# Otherwise this data won't match the rankings data used later
team_list_2022[3] = 'USA'
team_list_2022[25] = 'IR IRAN'
team_results_2022[3][0] = 'USA'
team_results_2022[25][0] = 'IR IRAN'